# كشف الأجسام باستخدام YOLO11n

<div dir="rtl">

يقوم هذا الدفتر بـ:

1. قراءة بيانات كشف الأجسام من Google Drive وفك ضغطها.
2. البحث تلقائيًا عن كل الصور وملفات التسميات المطابقة لها.
3. تقسيم البيانات إلى 70% تدريب و20% تحقق و10% اختبار.
4. إجراء Fine-Tuning لنموذج `yolo11n.pt`.
5. تقييم النموذج على مجموعة الاختبار.
6. إنشاء فيديو نتائج للمقطع من الدقيقة 4 إلى الدقيقة 9 مع صناديق كشف وأسماء عربية.

</div>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip -q install ultralytics arabic-reshaper python-bidi

## إعدادات المسارات
<div dir="rtl">
عدّل المسارات التالية قبل التشغيل.
</div>

In [ ]:
DATASET_ZIP = "/content/drive/MyDrive/dataset.zip"
VIDEO_PATH  = "/content/drive/MyDrive/test_video.mp4"
OUTPUT_DIR  = "/content/drive/MyDrive/yolo11n_outputs"

START_SECOND = 4 * 60
END_SECOND   = 9 * 60

TRAIN_RATIO = 0.70
VAL_RATIO   = 0.20
TEST_RATIO  = 0.10

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
assert abs(TRAIN_RATIO + VAL_RATIO + TEST_RATIO - 1) < 1e-6
assert os.path.exists(DATASET_ZIP), "ملف البيانات غير موجود."
assert os.path.exists(VIDEO_PATH), "فيديو الاختبار غير موجود."
print("المسارات صحيحة.")

## فك ضغط البيانات
<div dir="rtl">
سيبحث الدفتر عن الصور وملفات `txt` داخل كل المجلدات الفرعية تلقائيًا.
</div>

In [ ]:
!unzip -q "{DATASET_ZIP}" -d /content/yolo_dataset_raw

import glob, os

DATASET_ROOT = "/content/yolo_dataset_raw"

all_images = sorted(glob.glob(f"{DATASET_ROOT}/**/*.*", recursive=True))
all_images = [f for f in all_images if f.lower().endswith(
    (".jpg", ".jpeg", ".png", ".bmp", ".webp"))]

all_labels = sorted(glob.glob(f"{DATASET_ROOT}/**/*.txt", recursive=True))

print("عدد الصور المكتشفة:", len(all_images))
print("عدد ملفات التسميات:", len(all_labels))

if len(all_images) == 0:
    raise FileNotFoundError("لم يتم العثور على صور داخل الملف المضغوط.")

print("\nأمثلة للصور:")
for p in all_images[:5]:
    print(p)

print("\nأمثلة لملفات التسميات:")
for p in all_labels[:5]:
    print(p)

## تقسيم البيانات 70 / 20 / 10
<div dir="rtl">
نربط كل صورة بملف التسمية الذي يحمل نفس الاسم بدون امتداد، ثم نقسم الأزواج.
</div>

In [ ]:
import random, shutil

random.seed(42)

SPLIT_DIR = "/content/yolo_dataset_split"

for split in ["train", "val", "test"]:
    os.makedirs(f"{SPLIT_DIR}/images/{split}", exist_ok=True)
    os.makedirs(f"{SPLIT_DIR}/labels/{split}", exist_ok=True)

label_index = {}
for label_path in all_labels:
    base_name = os.path.splitext(os.path.basename(label_path))[0]
    label_index[base_name] = label_path

paired_images = []
missing_labels = []

for img_path in all_images:
    base_name = os.path.splitext(os.path.basename(img_path))[0]
    label_path = label_index.get(base_name)

    if label_path:
        paired_images.append((img_path, label_path))
    else:
        missing_labels.append(img_path)

print("عدد الصور المرتبطة بتسميات:", len(paired_images))
print("عدد الصور بدون تسميات:", len(missing_labels))

if len(paired_images) == 0:
    raise FileNotFoundError(
        "لم يتم العثور على ملفات تسميات مطابقة لأسماء الصور. "
        "تأكد أن اسم ملف txt نفس اسم الصورة بدون الامتداد."
    )

random.shuffle(paired_images)

n = len(paired_images)
n_train = int(n * TRAIN_RATIO)
n_val = int(n * VAL_RATIO)

train_pairs = paired_images[:n_train]
val_pairs = paired_images[n_train:n_train + n_val]
test_pairs = paired_images[n_train + n_val:]

splits = {
    "train": train_pairs,
    "val": val_pairs,
    "test": test_pairs
}

for split, pairs in splits.items():
    for img_path, label_path in pairs:
        img_dst = os.path.join(SPLIT_DIR, "images", split, os.path.basename(img_path))
        label_dst = os.path.join(SPLIT_DIR, "labels", split, os.path.basename(label_path))

        shutil.copy2(img_path, img_dst)
        shutil.copy2(label_path, label_dst)

print("عدد صور التدريب:", len(train_pairs))
print("عدد صور التحقق:", len(val_pairs))
print("عدد صور الاختبار:", len(test_pairs))

if missing_labels:
    print("\nأمثلة لصور بدون ملف تسمية:")
    for img_path in missing_labels[:5]:
        print("-", img_path)

## إنشاء data.yaml
<div dir="rtl">
إذا كان ملف `data.yaml` موجودًا في البيانات، سيتم استخدامه. وإلا أنشئ ملفًا جديدًا بعد تعديل أسماء الأصناف.
</div>

In [ ]:
yaml_files = glob.glob(f"{DATASET_ROOT}/**/data.yaml", recursive=True)

if yaml_files:
    import shutil
    DATA_YAML = "/content/data.yaml"
    shutil.copy2(yaml_files[0], DATA_YAML)
    print("تم استخدام ملف data.yaml الموجود.")
else:
    class_names = ["class_1", "class_2", "class_3", "class_4", "class_5"]

    DATA_YAML = "/content/data.yaml"

    yaml_content = f"""
path: {SPLIT_DIR}
train: images/train
val: images/val
test: images/test

names:
"""
    for i, name in enumerate(class_names):
        yaml_content += f"  {i}: {name}\n"

    with open(DATA_YAML, "w") as f:
        f.write(yaml_content)

    print("تم إنشاء ملف data.yaml جديد.")

!cat "{DATA_YAML}"

## تدريب YOLO11n
<div dir="rtl">
نبدأ من الأوزان المدرّبة مسبقًا وندرّب النموذج على بياناتنا.
</div>

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

model.train(
    data=DATA_YAML,
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    project=os.path.join(OUTPUT_DIR, "training"),
    name="yolo11n_fine_tune",
    exist_ok=True
)

## تقييم النموذج على الاختبار
<div dir="rtl">
نستخدم أفضل أوزان تم الحصول عليها أثناء التدريب لتقييم الأداء على 10% الاختبار.
</div>

In [ ]:
best_model_path = os.path.join(
    OUTPUT_DIR, "training", "yolo11n_fine_tune", "weights", "best.pt")

best_model = YOLO(best_model_path)

metrics = best_model.val(
    data=DATA_YAML,
    split="test",
    project=os.path.join(OUTPUT_DIR, "evaluation"),
    name="test_evaluation",
    exist_ok=True
)

print(metrics)

## فيديو النتائج بالعربية
<div dir="rtl">
نطبّق النموذج على الفيديو من الدقيقة 4 إلى الدقيقة 9 ونرسم الصناديق والأسماء العربية.
</div>

In [ ]:
import cv2
import numpy as np
import pandas as pd
import arabic_reshaper
from bidi.algorithm import get_display
from PIL import Image, ImageDraw, ImageFont

ARABIC_CLASS_NAMES = {
    "class_1": "الصنف الأول",
    "class_2": "الصنف الثاني",
    "class_3": "الصنف الثالث",
    "class_4": "الصنف الرابع",
    "class_5": "الصنف الخامس"
}

def get_arabic_name(name):
    return ARABIC_CLASS_NAMES.get(name, name)

font_path = "/content/ArabicFont.ttf"
if not os.path.exists(font_path):
    !wget -q -O "{font_path}" "https://github.com/google/fonts/raw/main/ofl/amiri/Amiri-Regular.ttf"

font = ImageFont.truetype(font_path, 32)

def draw_arabic_label(frame, text, x1, y1):
    img_pil = Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    draw = ImageDraw.Draw(img_pil)

    reshaped = arabic_reshaper.reshape(text)
    bidi_text = get_display(reshaped)

    bbox = draw.textbbox((0, 0), bidi_text, font=font)
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]

    y_text = max(0, y1 - text_h - 16)

    draw.rectangle([x1, y_text, x1 + text_w + 16, y_text + text_h + 12],
                   fill=(0, 0, 0))
    draw.text((x1 + 8, y_text + 4), bidi_text, font=font, fill=(0, 255, 0))

    return cv2.cvtColor(np.array(img_pil), cv2.COLOR_RGB2BGR)

ARABIC_OUTPUT_VIDEO = os.path.join(OUTPUT_DIR, "test_video_yolo11n_arabic.mp4")

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 30
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

start_frame = int(START_SECOND * fps)
end_frame = int(END_SECOND * fps)
cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(ARABIC_OUTPUT_VIDEO, fourcc, fps, (width, height))

frame_id = start_frame
detections_log = []

while True:
    ret, frame = cap.read()
    if not ret or frame_id >= end_frame:
        break

    result = best_model.predict(frame, imgsz=640, conf=0.25, verbose=False)[0]

    for box in result.boxes:
        x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
        class_id = int(box.cls[0])
        confidence = float(box.conf[0])

        english_name = result.names[class_id]
        arabic_name = get_arabic_name(english_name)
        label = f"{arabic_name}: {confidence:.2f}"

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
        frame = draw_arabic_label(frame, label, x1, y1)

        detections_log.append({
            "frame": frame_id,
            "second": frame_id / fps,
            "english_class": english_name,
            "arabic_class": arabic_name,
            "confidence": confidence,
            "x1": x1, "y1": y1, "x2": x2, "y2": y2
        })

    out.write(frame)
    frame_id += 1

cap.release()
out.release()

pd.DataFrame(detections_log).to_csv(
    os.path.join(OUTPUT_DIR, "video_detections_arabic.csv"),
    index=False, encoding="utf-8-sig")

print("تم إنشاء فيديو النتائج:")
print(ARABIC_OUTPUT_VIDEO)
print("عدد عمليات الكشف:", len(detections_log))

## عرض الفيديو
<div dir="rtl">
شغّل الخلية لمشاهدة الفيديو الناتج.
</div>

In [ ]:
from IPython.display import Video, display
display(Video(ARABIC_OUTPUT_VIDEO, embed=True, width=720))

## حفظ أفضل نموذج
<div dir="rtl">
نسخ أفضل أوزان إلى Drive.
</div>

In [ ]:
import shutil

final_weights = os.path.join(OUTPUT_DIR, "yolo11n_best.pt")
shutil.copy2(best_model_path, final_weights)

print("تم حفظ أفضل نموذج في:")
print(final_weights)